# A/A-тест: проверка системы сплитования

Источник данных: `simulator_20260620.feed_actions`. Берём период с 19 по 25 июня 2026 года и экспериментальные группы 2 и 3.

In [ ]:
import pandas as pd
import pandahouse
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
from getpass import getpass

db = "simulator_20260620"

connection = {
    "host": "http://clickhouse.lab.karpov.courses:8123",
    "database": db,
    "user": "student",
    "password": getpass("Пароль ClickHouse: "),
}

In [ ]:
test_query = """
SELECT
    count() AS rows_count,
    uniq(user_id) AS users_count
FROM {db}.feed_actions
WHERE toDate(time) BETWEEN '2026-06-19' AND '2026-06-25'
  AND exp_group IN (2, 3)
"""

pandahouse.read_clickhouse(test_query, connection=connection)

In [ ]:
query = """
SELECT
    exp_group,
    user_id,
    sum(action = 'like') AS likes,
    sum(action = 'view') AS views,
    likes / views AS ctr
FROM {db}.feed_actions
WHERE toDate(time) BETWEEN '2026-06-19' AND '2026-06-25'
  AND exp_group IN (2, 3)
GROUP BY
    exp_group,
    user_id
"""

df = pandahouse.read_clickhouse(query, connection=connection)
df.head()

In [ ]:
df.groupby("exp_group").agg(
    users=("user_id", "nunique"),
    mean_ctr=("ctr", "mean"),
)